# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

meta = dataset.metadata  # meta is a CroissantMetadata object
print(f"Dataset Name: {meta.name}\nDescription: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.


In [ ]:
# List all record sets and their fields, referencing by @id
record_sets = meta.record_sets
if not record_sets:
    print("No record sets found in the dataset.")
else:
    for rs in record_sets:
        print(f"---\nRecord Set Name: {rs.name}")
        print(f"Record Set @id: {rs.id}")
        print("Fields:")
        for f in rs.fields:
            print(f"  - Field Name: {f.name}, Field @id: {f.id}, DataType: {f.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.


In [ ]:
# Find all record set @ids for this dataset
record_sets = meta.record_sets
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}

for record_set in record_set_ids:
    records = list(dataset.records(record_set=record_set))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded record set @id: {record_set} with columns: {df.columns.tolist()}")
    else:
        print(f"No records found for record set @id: {record_set}")

# For demonstration, select the first available record set
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
    print(f"\nFirst 5 rows for {selected_record_set_id}:")
    display(dataframes[selected_record_set_id].head())
else:
    selected_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. For demonstration, we will filter by a numeric field if available.


In [ ]:
import numpy as np

if selected_record_set_id is not None:
    # Attempt to automatically find a numeric field (float or integer)
    candidate_columns = []
    for f in [rs for rs in meta.record_sets if rs.id==selected_record_set_id][0].fields:
        if f.data_type in ["Float", "Integer", "Number"]:
            candidate_columns.append(f.id)

    if candidate_columns:
        numeric_field = candidate_columns[0]  # Use the first numeric field found
        df = dataframes[selected_record_set_id]

        # Ensure data can be treated as numeric (convert if necessary)
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

        threshold = df[numeric_field].dropna().median()

        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try group by a categorical field if possible
        # Find the first non-numeric field
        group_field = None
        for f in [rs for rs in meta.record_sets if rs.id==selected_record_set_id][0].fields:
            if f.data_type in ["Text", "String"] and f.id in df.columns:
                group_field = f.id
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field} (showing first five groups):")
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA in this record set.")
else:
    print("No tabular data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id is not None and candidate_columns:
    field = candidate_columns[0]
    df = dataframes[selected_record_set_id].copy()
    df[field] = pd.to_numeric(df[field], errors='coerce')
    plt.figure(figsize=(8, 4))
    sns.histplot(df[field].dropna(), bins=30, kde=True, color='skyblue')
    plt.title(f"Distribution of {field} in {selected_record_set_id}")
    plt.xlabel(field)
    plt.ylabel("Count")
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, you have loaded the FAIR² Croissant-based dataset with `mlcroissant`, examined record sets and fields by their `@id`s, and performed some basic exploratory data analysis. You can now continue with advanced analyses or export the data for downstream bioinformatics or statistical workflows.
